# S7 — Explore the PushT dataset
Inspect dataset metadata, observation/action shapes, an RGB frame, and one expert trajectory. Execute with the project's `.venv` kernel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lerobot.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset('lerobot/pusht')
print('Episodes:', dataset.num_episodes)
print('Frames:', dataset.num_frames)
print('FPS:', dataset.fps)
for feature, specification in dataset.features.items():
    print(feature, specification)


In [ ]:
sample = dataset[0]
for name, value in sample.items():
    print(name, tuple(value.shape) if hasattr(value, 'shape') else type(value).__name__)
print('Action dimensions:', sample['action'].numel())


In [ ]:
image_keys = [k for k in sample if 'image' in k and hasattr(sample[k], 'shape')]
print('Image features:', image_keys)
if image_keys:
    image = sample[image_keys[0]].detach().cpu().numpy()
    if image.ndim == 3 and image.shape[0] in (1, 3, 4):
        image = np.transpose(image, (1, 2, 0))
    plt.figure(figsize=(6, 6))
    plt.imshow(np.clip(image, 0, 1) if image.dtype.kind == 'f' else image)
    plt.axis('off')
    plt.title('PushT expert demonstration — first observation')
    plt.show()


In [ ]:
episode_id = int(sample['episode_index'].item())
actions, states, timestamps = [], [], []
for idx in range(dataset.num_frames):
    row = dataset[idx]
    if int(row['episode_index'].item()) != episode_id:
        break
    actions.append(row['action'].detach().cpu().numpy())
    states.append(row['observation.state'].detach().cpu().numpy())
    timestamps.append(float(row['timestamp'].item()) if 'timestamp' in row else len(timestamps) / dataset.fps)
actions = np.stack(actions)
states = np.stack(states)
timestamps = np.asarray(timestamps) - timestamps[0]
print(f'Episode {episode_id}: {len(actions)} frames; actions {actions.shape}; states {states.shape}')
fig, ax = plt.subplots(figsize=(10, 4))
for dim in range(actions.shape[1]):
    ax.plot(timestamps, actions[:, dim], label=f'action[{dim}]')
ax.set(xlabel='Time (s)', ylabel='Action value', title='Expert actions across one episode')
ax.legend()
ax.grid(alpha=0.3)
plt.show()


In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(states[:, 0], states[:, 1], label='Agent trajectory')
plt.scatter(states[0, 0], states[0, 1], label='Start')
plt.scatter(states[-1, 0], states[-1, 1], marker='x', label='End')
plt.xlabel('Agent state[0]')
plt.ylabel('Agent state[1]')
plt.axis('equal')
plt.grid(alpha=0.3)
plt.legend()
plt.title('Expert agent trajectory (not T-block trajectory)')
plt.show()


## End-of-week notes
Write your own observations after completing the week's tasks.